# AfriVoices — SOUMISSION GREEDY (sans modèle de langue)

Trois objectifs :
1. **Mesurer la contribution réelle du KenLM au leaderboard** : `contribution = LB_greedy − 0.39923`
   (même modèle v9-2, même moteur certifié — seul le décodage change : argmax CTC).
2. Servir d'**étalon** pour juger tout LM externe : un LM ne vaut que s'il bat 0.39923,
   et l'écart greedy↔0.39923 borne ce qu'un LM peut apporter.
3. Base 100 % indépendante de pyctcdecode/kenlm (aucune dépendance à installer).

Moteur identique au générateur certifié : décodage direct pleine longueur, troncature
anti-padding, secours fp16 si OOM, reprise auto. **Attendu : nettement au-dessus de
0.39923** — c'est normal, c'est la valeur du LM qu'on mesure.

**Résultat obtenu : ≈ 0.45** → contribution du KenLM v2 ≈ −0.055
(0.45 glouton → 0.39477 finale). Sert la décomposition du RAPPORT (Partie II §1).

Ordre : §2 → §3 → §4 → §5 → §6. Durée ~30-45 min sur A100 (aucun goulot CPU).

## 1. Dépendances — aucune

In [ ]:
print("✓ rien à installer (torch/transformers/soundfile/librosa préinstallés) — continue en §2")

## 2. CONFIG

In [ ]:
# ============================================================
MODEL_NAME = "baobab-asr-v9-2"
TAG = "v9_2_greedy"
# ============================================================

from google.colab import drive
drive.mount("/content/drive")
import os
BASE  = "/content/drive/MyDrive/afrivoices"
MODEL = f"{BASE}/{MODEL_NAME}"
PARTIEL = f"{BASE}/submission_{TAG}_partiel.csv"
FINAL   = f"{BASE}/submission_{TAG}.csv"
assert os.path.isdir(MODEL), f"modèle absent: {MODEL}"
print(f"modèle: {MODEL_NAME} | décodage: GREEDY (argmax, sans LM)")
print(f"sortie: {FINAL}")

## 3. Modèle + decode_robuste

In [ ]:
import torch, io, base64, contextlib, numpy as np, soundfile as sf, librosa, tempfile
from transformers import Wav2Vec2BertForCTC, Wav2Vec2BertProcessor

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device, "" if device == "cuda" else "⚠️ GPU recommandé")

model = Wav2Vec2BertForCTC.from_pretrained(MODEL, local_files_only=True).to(device).eval()
processor = Wav2Vec2BertProcessor.from_pretrained(MODEL, local_files_only=True)

def decode_robuste(b):
    if not b: raise ValueError("vide")
    try: arr, sr = sf.read(io.BytesIO(b), dtype="float32")
    except Exception:
        try: arr, sr = sf.read(io.BytesIO(base64.b64decode(b)), dtype="float32")
        except Exception:
            with tempfile.NamedTemporaryFile(suffix=".wav", delete=False) as tf:
                try: tf.write(base64.b64decode(b))
                except Exception: tf.write(b)
                p = tf.name
            arr, sr = librosa.load(p, sr=16000, mono=True); os.unlink(p)
            return arr.astype(np.float32)
    if arr.ndim > 1: arr = arr.mean(axis=1)
    if sr != 16000: arr = librosa.resample(arr, orig_sr=sr, target_sr=16000)
    return arr.astype(np.float32)

print("✓ prêt")

## 4. Données de test

In [ ]:
import glob
TEST_CANDIDATES = [f"{BASE}/test", "/content/test_data"]
parquets = []
for td in TEST_CANDIDATES:
    parquets = sorted(glob.glob(f"{td}/**/*.parquet", recursive=True))
    if parquets:
        TEST_DIR = td; break
if not parquets:
    print("test absent localement -> téléchargement HF...")
    from huggingface_hub import snapshot_download
    TEST_DIR = "/content/test_data"
    snapshot_download("digitalumuganda/anv-test-data-nt", repo_type="dataset", local_dir=TEST_DIR)
    parquets = sorted(glob.glob(f"{TEST_DIR}/**/*.parquet", recursive=True))
print(f"{len(parquets)} parquets test dans {TEST_DIR}")
assert parquets, "aucun parquet test trouvé"

## 5. Inférence greedy (direct pleine longueur, troncature anti-padding, reprise auto)

In [ ]:
import pandas as pd, time
from tqdm.auto import tqdm

MAX_SEC_BATCH = 160.0

def _forward(feats, fp16=False):
    ctx = torch.autocast("cuda", dtype=torch.float16) if (fp16 and device == "cuda") else contextlib.nullcontext()
    with torch.no_grad(), ctx:
        return model(**{k: v.to(device) for k, v in feats.items()}).logits.float().cpu().numpy()

def _solo(arr):
    fe = processor.feature_extractor([arr], sampling_rate=16000, return_tensors="pt")
    try:
        return _forward(fe)[0]
    except RuntimeError:
        torch.cuda.empty_cache()
        return _forward(fe, fp16=True)[0]

def logits_batch(arrs):
    """Logits TRONQUÉS à la vraie longueur (anti-padding) ; item par item si OOM."""
    order = sorted(range(len(arrs)), key=lambda k: len(arrs[k]))
    out = [None] * len(arrs); i = 0
    while i < len(order):
        j = i + 1
        while j < len(order) and (j - i + 1) * (len(arrs[order[j]]) / 16000.0) <= MAX_SEC_BATCH:
            j += 1
        idxs = order[i:j]
        feats = processor.feature_extractor([arrs[k] for k in idxs], sampling_rate=16000,
                                            return_tensors="pt", padding=True)
        try:
            lg = _forward(feats)
            am = feats.get("attention_mask")
            T_out = lg.shape[1]
            if am is not None:
                fr = am.sum(-1).numpy().astype(float); fmax = float(am.shape[1])
            else:
                fr = np.array([len(arrs[k]) for k in idxs], dtype=float); fmax = fr.max()
            for bi, k in enumerate(idxs):
                n_true = max(1, int(round(T_out * fr[bi] / fmax)))
                out[k] = lg[bi, :min(n_true, T_out)]
        except RuntimeError:
            torch.cuda.empty_cache()
            for k in idxs:
                out[k] = _solo(arrs[k])
        i = j
    return out

done = {}
if os.path.exists(PARTIEL):
    dfp = pd.read_csv(PARTIEL)
    done = dict(zip(dfp["id"].astype(str), zip(dfp["language"], dfp["transcription"])))
    print("reprise :", len(done), "déjà transcrits")

rows = [{"id": k, "language": v[0], "transcription": v[1]} for k, v in done.items()]
t0 = time.time()

for pi, pq in enumerate(tqdm(parquets, desc="Inférence greedy", unit="pq")):
    df = pd.read_parquet(pq)
    items = []   # (rid, lang, arr)
    for _, r in df.iterrows():
        rid = str(r["id"])
        if rid in done: continue
        lang = r.get("language")
        b = r["audio"].get("bytes") if isinstance(r["audio"], dict) else r["audio"]
        try:
            items.append((rid, lang, decode_robuste(b)))
        except Exception:
            rows.append({"id": rid, "language": lang, "transcription": "_"})
    if items:
        lgs = logits_batch([a for _, _, a in items])
        for (rid, lang, _), lg in zip(items, lgs):
            try:
                txt = processor.decode(lg.argmax(-1).tolist()).strip()
            except Exception:
                txt = "_"
            rows.append({"id": rid, "language": lang, "transcription": txt or "_"})
    if (pi + 1) % 10 == 0 or pi == len(parquets) - 1:
        pd.DataFrame(rows).to_csv(PARTIEL, index=False)
        print(f"parquet {pi + 1}/{len(parquets)} | {len(rows)} transcrits | {(time.time() - t0)/60:.0f} min")

print("✓ inférence greedy terminée :", len(rows))

## 6. Assemblage final + asserts + CSV

In [ ]:
import pandas as pd
sub = pd.DataFrame(rows)
sub["transcription"] = (sub["transcription"].astype(str)
                        .str.replace(r"[\r\n]+", " ", regex=True).str.strip())
sub.loc[sub["transcription"].isin(["", "nan", "None"]), "transcription"] = "_"
sub["id"] = sub["id"].astype(str)
sub = sub.drop_duplicates(subset="id", keep="first")[["id", "language", "transcription"]]

assert list(sub.columns) == ["id", "language", "transcription"]
assert len(sub) == 41733, f"attendu 41733, obtenu {len(sub)}"
assert sub["id"].is_unique
assert sub["language"].notna().all()
assert (sub["transcription"].str.len() > 0).all()

sub.to_csv(FINAL, index=False)
print("✅ soumission écrite ->", FINAL)
print(sub["language"].value_counts())

## 7. Après la soumission

- Note `LB_greedy`. **Contribution du KenLM v2 = LB_greedy − 0.39923.**
- Grille de lecture pour le LM externe : il doit battre **0.39923** (pas le greedy) pour
  valoir une adoption ; l'écart greedy↔0.39923 borne ce qu'un LM peut apporter.
- Ce CSV est aussi une soumission de secours 100 % indépendante de pyctcdecode/kenlm.